In [11]:
# ============================================================
# [CELL 00] — CONFIG & IMPORTS
# Projet POKEMON — Référentiel Universel FR
# Prismora Solutions — v2.0 — 2026
# Notebook : 01_SETUP_BDD.ipynb
# ============================================================

import requests
import pandas as pd
import json
import os
import time
from pathlib import Path
from IPython.display import display

# Répertoire racine = dossier du notebook
ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config" / "sources.json"

# Arborescence complète du projet
DOSSIERS = [
    "config",
    "data/raw",
    "data/processed",
    "data/exports/print",
    "assets/images/pocket",
    "assets/images/tcg",
    "assets/images/pokemon",
]

for folder in DOSSIERS:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

SOURCE_PRIMARY   = CONFIG["sources"]["primary"]
SOURCE_SECONDARY = CONFIG["sources"]["secondary"]
SOURCE_TERTIARY  = CONFIG["sources"]["tertiary"]

# Contrôle
print("=" * 55)
print("  PROJET POKEMON — Référentiel Universel FR")
print("  Prismora Solutions — 01_SETUP_BDD.ipynb")
print("=" * 55)
print(f"\n📁 Racine projet : {ROOT}")
print(f"\n📂 Arborescence :")
for folder in DOSSIERS:
    p = ROOT / folder
    print(f"   {'✅' if p.exists() else '❌'} {folder}")
print(f"\n⚙️  Config attendue : {CONFIG_PATH}")
print(f"   {'✅ Présente' if CONFIG_PATH.exists() else '❌ MANQUANTE — créer config/sources.json'}")

  PROJET POKEMON — Référentiel Universel FR
  Prismora Solutions — 01_SETUP_BDD.ipynb

📁 Racine projet : i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd

📂 Arborescence :
   ✅ config
   ✅ data/raw
   ✅ data/processed
   ✅ data/exports/print
   ✅ assets/images/pocket
   ✅ assets/images/tcg
   ✅ assets/images/pokemon

⚙️  Config attendue : i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd\config\sources.json
   ✅ Présente


In [12]:
# ============================================================
# [CELL 01] — CHARGEMENT CONFIG SOURCES
# Changer de source = modifier sources.json uniquement
# ============================================================

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

# Sources
SOURCE_PRIMARY = CONFIG["sources"]["primary"]
SOURCE_SECONDARY = CONFIG["sources"]["secondary"]
SOURCE_FALLBACK = CONFIG["sources"]["fallback"]

BASE_URL = SOURCE_PRIMARY["base_url"]

# Contrôle
print("✅ Config chargée\n")
print(f"🥇 Primary   : {SOURCE_PRIMARY['name']}")
print(f"   URL       : {SOURCE_PRIMARY['base_url']}")
print(f"   Actif     : {SOURCE_PRIMARY['active']}")
print(f"\n🥈 Secondary : {SOURCE_SECONDARY['name']}")
print(f"   URL       : {SOURCE_SECONDARY['base_url']}")
print(f"   Actif     : {SOURCE_SECONDARY['active']}")
print(f"\n🥉 Fallback  : {SOURCE_FALLBACK['name']}")
print(f"   Actif     : {SOURCE_FALLBACK['active']}")
print(f"\n📄 Outputs attendus :")
for k, v in CONFIG["output"].items():
    print(f"   {k:15s} → {v}")

✅ Config chargée

🥇 Primary   : TCGdex API (FR natif)
   URL       : https://api.tcgdex.net/v2/fr
   Actif     : True

🥈 Secondary : flibustier/pokemon-tcg-pocket-database
   URL       : https://raw.githubusercontent.com/flibustier/pokemon-tcg-pocket-database/main/dist
   Actif     : True

🥉 Fallback  : LimitlessTCG
   Actif     : False

📄 Outputs attendus :
   sets_master     → data/processed/sets_master.csv
   cards_pocket    → data/processed/cards_pocket_fr.csv
   cards_fr_en     → data/processed/cards_fr_en.csv
   pokemon_master  → data/processed/pokemon_master.csv
   images_pocket   → assets/images/pocket
   images_tcg      → assets/images/tcg
   images_pokemon  → assets/images/pokemon


In [13]:
# ============================================================
# [CELL 02] — RÉCUPÉRATION DES SETS TCG POCKET
# Double source : TCGdex FR + flibustier
# ============================================================

# --- SOURCE 1 : TCGdex FR ---
url_tcgdex = BASE_URL + SOURCE_PRIMARY["sets_endpoint"]
resp = requests.get(url_tcgdex, timeout=15)
resp.raise_for_status()
sets_tcgdex = resp.json()

# --- SOURCE 2 : flibustier ---
url_flib = SOURCE_SECONDARY["base_url"] + SOURCE_SECONDARY["sets_endpoint"]
resp2 = requests.get(url_flib, timeout=15)
resp2.raise_for_status()
sets_flib_raw = resp2.json()

# flibustier structure : {"A": [...], "B": [...]} → on aplatit
sets_flib = []
for serie, liste in sets_flib_raw.items():
    for s in liste:
        s["serie"] = serie
        sets_flib.append(s)

# Sauvegarde brute
with open(ROOT / "data/raw/sets_tcgdex.json", "w", encoding="utf-8") as f:
    json.dump(sets_tcgdex, f, ensure_ascii=False, indent=2)
with open(ROOT / "data/raw/sets_flibustier.json", "w", encoding="utf-8") as f:
    json.dump(sets_flib, f, ensure_ascii=False, indent=2)

# Contrôle
print(f"✅ TCGdex    : {len(sets_tcgdex)} sets récupérés")
print(f"✅ flibustier : {len(sets_flib)} sets Pocket récupérés")
print(f"\n🔍 Structure flibustier (1 set) :")
print(json.dumps(sets_flib[0], ensure_ascii=False, indent=2))
print(f"\n🔍 Structure TCGdex (1 set) :")
print(json.dumps(sets_tcgdex[0], ensure_ascii=False, indent=2))

✅ TCGdex    : 191 sets récupérés
✅ flibustier : 19 sets Pocket récupérés

🔍 Structure flibustier (1 set) :
{
  "code": "A1",
  "releaseDate": "2024-10-30",
  "count": 286,
  "name": {
    "en": "Genetic Apex",
    "fr": "Puissance Génétique",
    "de": "Unschlagbare Gene",
    "zh": "最強的基因",
    "pt": "Dominação Genética",
    "es": "Genes Formidables",
    "it": "Geni Supremi",
    "ja": "最強の遺伝子",
    "ko": "최강의유전자"
  },
  "packs": [
    "Charizard",
    "Mewtwo",
    "Pikachu"
  ],
  "serie": "A"
}

🔍 Structure TCGdex (1 set) :
{
  "id": "base1",
  "name": "Set de Base",
  "cardCount": {
    "total": 102,
    "official": 102
  }
}


In [14]:
# ============================================================
# [CELL 03] — CONSTRUCTION sets_master.csv
# flibustier = référence Pocket | TCGdex = complément IDs
# ============================================================

# Index TCGdex par nom EN pour croisement
tcgdex_index = {}
for s in sets_tcgdex:
    tcgdex_index[s["name"].lower().strip()] = s["id"]

# Construction de la table sets_master depuis flibustier
rows = []
for s in sets_flib:
    nom_en = s["name"].get("en", "")
    # Recherche ID TCGdex correspondant
    id_tcgdex = tcgdex_index.get(nom_en.lower().strip(), "")

    rows.append({
        "id_set":        f"SET-PKT-{s['code']}",
        "code_set":      s["code"],
        "serie":         s["serie"],
        "jeu":           "pocket",
        "nom_fr":        s["name"].get("fr", ""),
        "nom_en":        nom_en,
        "nom_de":        s["name"].get("de", ""),
        "nom_ja":        s["name"].get("ja", ""),
        "date_sortie":   s.get("releaseDate", ""),
        "nb_cartes":     s.get("count", 0),
        "packs":         ", ".join(s.get("packs", [])),
        "id_tcgdex":     id_tcgdex,
    })

df_sets = pd.DataFrame(rows).sort_values("date_sortie").reset_index(drop=True)

# Export CSV
out = ROOT / CONFIG["output"]["sets_master"]
df_sets.to_csv(out, index=False, encoding="utf-8-sig")

# Contrôle
print(f"✅ {len(df_sets)} sets Pocket construits")
print(f"💾 {out}\n")
print(f"🔍 IDs TCGdex trouvés : {df_sets['id_tcgdex'].astype(bool).sum()} / {len(df_sets)}")
print(f"\n📋 sets_master.csv :")
display(df_sets[["id_set", "code_set", "serie", "nom_fr", "nom_en", "date_sortie", "nb_cartes", "id_tcgdex"]])

✅ 19 sets Pocket construits
💾 i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd\data\processed\sets_master.csv

🔍 IDs TCGdex trouvés : 0 / 19

📋 sets_master.csv :


,id_set,code_set,serie,nom_fr,nom_en,date_sortie,nb_cartes,id_tcgdex
0,SET-PKT-A1,A1,A,Puissance Génétique,Genetic Apex,2024-10-30,286,
1,SET-PKT-PROMO-A,PROMO-A,A,,Promo A,2024-10-30,117,
2,SET-PKT-A1a,A1a,A,L'Île Fabuleuse,Mythical Island,2024-12-17,85,
3,SET-PKT-A2,A2,A,,Space-Time Smackdown,2025-01-29,207,
4,SET-PKT-A2a,A2a,A,Lumière Triomphante,Triumphant Light,2025-02-28,96,
5,SET-PKT-A2b,A2b,A,,Shining Revelry,2025-03-27,111,
6,SET-PKT-A3,A3,A,,Celestial Guardians,2025-04-30,239,
7,SET-PKT-A3a,A3a,A,,Extradimensional Crisis,2025-05-29,103,
8,SET-PKT-A3b,A3b,A,,Eevee Grove,2025-06-26,107,
9,SET-PKT-A4,A4,A,,Wisdom of Sea and Sky,2025-07-30,241,


In [15]:
# ============================================================
# [CELL 04] — CONSTRUCTION sets_master.csv
# Sources : flibustier (référence) + TCGdex (noms FR)
#           + pokekalos.fr (noms FR fallback + nb_cartes + slugs)
# ============================================================

import re
from bs4 import BeautifulSoup

# --- SOURCE 2 : flibustier (référence sets Pocket) ---
url_flib = SOURCE_SECONDARY["base_url"] + SOURCE_SECONDARY["sets_endpoint"]
resp = requests.get(url_flib, timeout=15)
resp.raise_for_status()
sets_flib_raw = resp.json()
sets_flib = []
for serie, liste in sets_flib_raw.items():
    for s in liste:
        s["serie"] = serie
        sets_flib.append(s)

# --- SOURCE 1 : TCGdex FR (déjà chargé en CELL 02) ---
tcgdex_by_code = {s["id"]: s["name"] for s in sets_tcgdex}
MAP_CODES = {"PROMO-A": "P-A", "PROMO-B": "P-B"}

# --- SOURCE 3 : pokekalos.fr (noms FR + slugs + nb_cartes) ---
url_pokekalos = SOURCE_TERTIARY["base_url"] + SOURCE_TERTIARY["sets_endpoint"]
resp3 = requests.get(url_pokekalos, timeout=15)
resp3.raise_for_status()
soup = BeautifulSoup(resp3.text, "html.parser")
liens_extensions = [l for l in soup.find_all("a", href=True) if "extensions/" in l.get("href", "")]

sets_pokekalos_noms  = {}
sets_pokekalos_slugs = {}

for l in liens_extensions:
    text = l.get_text(strip=True)
    href = l.get("href", "")
    match = re.match(r'\[([A-Z0-9a-z\-]+)\]\s*(.+)', text)
    if match:
        code   = match.group(1).strip()
        nom_fr = match.group(2).strip()
        # Extraction slug depuis href — dernière partie avant .html
        slug   = href.split("extensions/")[-1].replace(".html", "").strip("/")
        sets_pokekalos_noms[code]  = nom_fr
        sets_pokekalos_slugs[code] = slug
    elif text.strip() in ["Promo-A", "Promo-B"]:
        code = "PROMO-A" if "A" in text else "PROMO-B"
        slug = href.split("extensions/")[-1].replace(".html", "").strip("/")
        sets_pokekalos_noms[code]  = text.strip()
        sets_pokekalos_slugs[code] = slug

# Scraping nb_cartes réel sur pokekalos (via slugs récupérés dynamiquement)
print("🔍 Scraping nb_cartes via pokekalos.fr ...")
sets_pokekalos_nb = {}
for code, slug in sets_pokekalos_slugs.items():
    url = f"https://www.pokekalos.fr/jeux/mobile/pocket/cartodex/extensions/{slug}.html"
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        s = BeautifulSoup(r.text, "html.parser")
        match = re.search(r'Nombre de cartes\s*:\s*(\d+)', s.get_text())
        sets_pokekalos_nb[code] = int(match.group(1)) if match else None
    except:
        sets_pokekalos_nb[code] = None
    time.sleep(0.3)
print(f"✅ nb_cartes récupérés pour {sum(1 for v in sets_pokekalos_nb.values() if v)} / {len(sets_pokekalos_slugs)} sets\n")

# Sauvegardes brutes
with open(ROOT / "data/raw/sets_flibustier.json", "w", encoding="utf-8") as f:
    json.dump(sets_flib, f, ensure_ascii=False, indent=2)
with open(ROOT / "data/raw/sets_pokekalos.json", "w", encoding="utf-8") as f:
    json.dump({
        "noms":   sets_pokekalos_noms,
        "slugs":  sets_pokekalos_slugs,
        "nb":     {k: str(v) for k, v in sets_pokekalos_nb.items()}
    }, f, ensure_ascii=False, indent=2)

# --- FUSION ---
rows = []
for s in sets_flib:
    code        = s["code"]
    code_tcgdex = MAP_CODES.get(code, code)

    # Nom FR — priorité : TCGdex > pokekalos > flibustier
    nom_fr_tcgdex    = tcgdex_by_code.get(code_tcgdex, "")
    nom_fr_pokekalos = sets_pokekalos_noms.get(code, "")
    nom_fr_flib      = s["name"].get("fr", "")
    if nom_fr_tcgdex:
        nom_fr, source_nom = nom_fr_tcgdex, "tcgdex"
    elif nom_fr_pokekalos:
        nom_fr, source_nom = nom_fr_pokekalos, "pokekalos"
    elif nom_fr_flib:
        nom_fr, source_nom = nom_fr_flib, "flibustier"
    else:
        nom_fr, source_nom = "", "manquant"

    # nb_cartes — pokekalos est la source de vérité
    nb_pokekalos = sets_pokekalos_nb.get(code)
    nb_flib      = s.get("count", 0)
    nb_cartes    = nb_pokekalos if nb_pokekalos is not None else nb_flib

    rows.append({
        "id_set":          f"SET-PKT-{code}",
        "code_set":        code,
        "code_tcgdex":     code_tcgdex,
        "serie":           s["serie"],
        "jeu":             "pocket",
        "nom_fr":          nom_fr,
        "nom_en":          s["name"].get("en", ""),
        "nom_ja":          s["name"].get("ja", ""),
        "date_sortie":     s.get("releaseDate", ""),
        "nb_cartes":       nb_cartes,
        "packs":           ", ".join(s.get("packs", [])),
        "slug_pokekalos":  sets_pokekalos_slugs.get(code, ""),
        "source_nom_fr":   source_nom,
        "source_nb":       "pokekalos" if nb_pokekalos else "flibustier",
    })

df_sets = pd.DataFrame(rows).sort_values("date_sortie").reset_index(drop=True)

# Export CSV
out = ROOT / CONFIG["output"]["sets_master"]
df_sets.to_csv(out, index=False, encoding="utf-8-sig")

# Contrôle
print(f"✅ {len(df_sets)} sets Pocket construits")
print(f"💾 {out}\n")
print(f"🇫🇷 Noms FR par source :")
display(df_sets.groupby("source_nom_fr").size().reset_index(name="nb_sets"))
manquants = df_sets[df_sets["nom_fr"] == ""]
if len(manquants):
    print(f"\n⚠️ Noms FR manquants : {list(manquants['code_set'])}")
else:
    print(f"\n✅ Tous les noms FR renseignés")
print(f"\n📋 sets_master.csv final :")
display(df_sets[["code_set", "nom_fr", "nom_en", "date_sortie",
                  "nb_cartes", "slug_pokekalos", "source_nom_fr"]])
print(f"\n📊 Total cartes : {df_sets['nb_cartes'].sum()}")
print(f"\n🔁 Pense à resync le Google Sheets sets_master !")

🔍 Scraping nb_cartes via pokekalos.fr ...
✅ nb_cartes récupérés pour 19 / 19 sets

✅ 19 sets Pocket construits
💾 i:\Drive partagés\Prismora - Drive Principal\01_Clients\pokemon-bdd\data\processed\sets_master.csv

🇫🇷 Noms FR par source :


,source_nom_fr,nb_sets
0,pokekalos,4
1,tcgdex,15



✅ Tous les noms FR renseignés

📋 sets_master.csv final :


,code_set,nom_fr,nom_en,date_sortie,nb_cartes,slug_pokekalos,source_nom_fr
0,A1,Puissance Génétique,Genetic Apex,2024-10-30,286,puissance-genetique,tcgdex
1,PROMO-A,Promo-A,Promo A,2024-10-30,117,promo-a,tcgdex
2,A1a,L’Île Fabuleuse,Mythical Island,2024-12-17,86,l-ile-fabuleuse,tcgdex
3,A2,Choc Spatio-Temporel,Space-Time Smackdown,2025-01-29,207,choc-spatio-temporel,tcgdex
4,A2a,Lumière Triomphale,Triumphant Light,2025-02-28,96,lumiere-triomphale,tcgdex
5,A2b,Réjouissances Rayonnantes,Shining Revelry,2025-03-27,111,rejouissances-rayonnantes,tcgdex
6,A3,Gardiens Astraux,Celestial Guardians,2025-04-30,239,gardiens-astraux,tcgdex
7,A3a,Crise Interdimensionnelle,Extradimensional Crisis,2025-05-29,103,crise-interdimensionnelle,tcgdex
8,A3b,La Clairière d'Évoli,Eevee Grove,2025-06-26,107,la-clairiere-d-evoli,tcgdex
9,A4,Sagesse Entre Ciel et Mer,Wisdom of Sea and Sky,2025-07-30,241,sagesse-entre-ciel-et-mer,tcgdex



📊 Total cartes : 3289

🔁 Pense à resync le Google Sheets sets_master !
